# Analisis rapido de nulos en ventas

Este notebook analiza el archivo **Video Games Sales (1980-2024) - Curated Clean.csv** con foco en:

1. Porcentaje de nulos por ano en ventas regionales y ventas totales.
2. Consolas/plataformas con mayor porcentaje de nulos en esas variables.
3. Registros repetidos por `game_title`-`platform` y diferencias entre ellos.

**Criterio importante:** para evitar porcentajes artificiales de 100%, los calculos de nulos por ano/plataforma se hacen solo sobre filas con al menos una venta reportada en estas columnas:
- `total_sales_final`
- `north_america_sales`
- `japan_sales`
- `europe_sales`
- `other_regions_sales`

In [6]:
import pandas as pd

csv_path = "Video Games Sales (1980-2024) - Curated Clean.csv"
df = pd.read_csv(csv_path)

# Usamos la variable de ventas totales consolidada + ventas regionales
sales_cols = [
    "total_sales_final",
    "north_america_sales",
    "japan_sales",
    "europe_sales",
    "other_regions_sales",
]

required_cols = sales_cols + ["release_year", "platform", "game_title"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"No existen columnas esperadas: {missing_cols}")

for c in sales_cols + ["release_year"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Mascara: fila con al menos una venta informada (evita grupos 100% vacios)
df["has_any_sales"] = df[sales_cols].notna().any(axis=1)

print(f"Filas totales: {len(df):,}")
print(f"Filas con alguna venta reportada: {df['has_any_sales'].sum():,}")
print("Columnas de ventas analizadas:", sales_cols)

Filas totales: 64,016
Filas con alguna venta reportada: 18,922
Columnas de ventas analizadas: ['total_sales_final', 'north_america_sales', 'japan_sales', 'europe_sales', 'other_regions_sales']


In [7]:
# 1) Porcentaje de nulos por ano (solo filas con alguna venta reportada)
year_base = df.dropna(subset=["release_year"]).copy()
year_base["release_year"] = year_base["release_year"].astype(int)
year_base = year_base[year_base["has_any_sales"]].copy()

year_stats = (
    year_base.groupby("release_year")[sales_cols]
    .apply(lambda g: pd.Series({
        "null_pct": g.isna().sum().sum() / (g.shape[0] * len(sales_cols)) * 100,
        "n_games": g.shape[0],
    }))
    .reset_index()
    .sort_values(["null_pct", "release_year"], ascending=[False, True])
)

max_year_pct = year_stats["null_pct"].max()
years_with_max = year_stats.loc[year_stats["null_pct"] == max_year_pct, "release_year"].sort_values()

print("Top 10 anos con mayor porcentaje de nulos (ventas regionales + total_sales_final):")
display(year_stats.head(10))

print(
    f"Mayor porcentaje de nulos por ano (filtrando filas totalmente vacias): {max_year_pct:.2f}%"
)
print("Anos exactos con el maximo:", years_with_max.tolist())

Top 10 anos con mayor porcentaje de nulos (ventas regionales + total_sales_final):


,release_year,null_pct,n_games
42,2019,58.857143,35.0
14,1991,48.000000,20.0
17,1994,46.391753,97.0
43,2020,45.714286,28.0
15,1992,45.217391,23.0
16,1993,43.111111,45.0
18,1995,39.085714,175.0
8,1985,35.000000,4.0
41,2018,34.757576,660.0
9,1986,34.545455,11.0


Mayor porcentaje de nulos por ano (filtrando filas totalmente vacias): 58.86%
Anos exactos con el maximo: [2019]


In [8]:
# 2) Plataformas con mayor porcentaje de nulos (solo filas con alguna venta reportada)
platform_base = df[df["has_any_sales"]].copy()

platform_null_pct = (
    platform_base.groupby("platform")[sales_cols]
    .apply(lambda g: pd.Series({
        "null_pct": g.isna().sum().sum() / (g.shape[0] * len(sales_cols)) * 100,
        "n_games": g.shape[0],
    }))
    .reset_index()
    .sort_values(["null_pct", "n_games", "platform"], ascending=[False, False, True])
)

print("Top 15 plataformas con mayor porcentaje de nulos (sin filas totalmente vacias):")
display(platform_null_pct.head(15))

max_platform_pct = platform_null_pct["null_pct"].max()
platforms_with_max = platform_null_pct.loc[platform_null_pct["null_pct"] == max_platform_pct, "platform"].tolist()
print(f"Mayor porcentaje de nulos por plataforma: {max_platform_pct:.2f}%")
print("Plataformas con ese maximo:", platforms_with_max)

print("\nTop 15 plataformas con mayor porcentaje de nulos (minimo 20 juegos):")
display(
    platform_null_pct
    .query("n_games >= 20")
    .head(15)
)

Top 15 plataformas con mayor porcentaje de nulos (sin filas totalmente vacias):


,platform,null_pct,n_games
1,3DO,60.000000,4.0
18,PCE,60.000000,2.0
10,GG,60.000000,1.0
19,PCFX,60.000000,1.0
14,NG,51.666667,12.0
31,WS,51.428571,7.0
27,SAT,50.285714,175.0
5,GB,50.000000,46.0
29,SNES,48.223350,197.0
28,SCD,48.000000,5.0


Mayor porcentaje de nulos por plataforma: 60.00%
Plataformas con ese maximo: ['3DO', 'PCE', 'GG', 'PCFX']

Top 15 plataformas con mayor porcentaje de nulos (minimo 20 juegos):


,platform,null_pct,n_games
27,SAT,50.285714,175.0
5,GB,50.000000,46.0
29,SNES,48.223350,197.0
3,DC,46.530612,49.0
26,PSV,46.058824,680.0
25,PSP,41.656716,1340.0
2,3DS,38.576512,562.0
9,GEN,38.518519,27.0
17,PC,37.089744,1560.0
4,DS,35.918197,2396.0


In [9]:
# 3) Duplicados por game_title-platform y diferencias entre filas del grupo
key_cols = ["game_title", "platform"]
dup_mask = df.duplicated(subset=key_cols, keep=False)
dup_df = df.loc[dup_mask].copy()

print(f"Registros involucrados en duplicados game_title-platform: {len(dup_df):,}")
print(f"Cantidad de pares duplicados unicos: {dup_df.groupby(key_cols).ngroups:,}")

def describe_group_differences(group):
    diff_cols = []
    for col in group.columns:
        if col in key_cols:
            continue
        # Cuenta valores distintos considerando nulos como categoria
        if group[col].astype(str).nunique(dropna=False) > 1:
            diff_cols.append(col)

    return pd.Series({
        "rows_in_group": len(group),
        "different_columns_count": len(diff_cols),
        "different_columns": ", ".join(diff_cols) if diff_cols else "(sin diferencias)",
    })

dup_summary = (
    dup_df.groupby(key_cols, as_index=False)
    .apply(describe_group_differences, include_groups=False)
    .reset_index(drop=True)
    .sort_values(["different_columns_count", "rows_in_group"], ascending=[False, False])
)

print("Top 20 pares duplicados con mas diferencias entre filas:")
display(dup_summary.head(20))

# Muestra de detalle para inspeccion manual
if not dup_summary.empty:
    ex_title = dup_summary.iloc[0]["game_title"]
    ex_platform = dup_summary.iloc[0]["platform"]
    print(f"\nEjemplo detallado del par con mas diferencias: {ex_title} | {ex_platform}")
    display(
        dup_df[(dup_df["game_title"] == ex_title) & (dup_df["platform"] == ex_platform)]
        .sort_index()
    )

Registros involucrados en duplicados game_title-platform: 449
Cantidad de pares duplicados unicos: 224
Top 20 pares duplicados con mas diferencias entre filas:


,game_title,platform,rows_in_group,different_columns_count,different_columns
222,Zoo Tycoon DS,DS,2,18,"genre, publisher, developer, critic_score, tot..."
201,Transformers: The Game,PSP,2,17,"genre, publisher, developer, critic_score, tot..."
100,Konami Krazy Racers,GBA,2,16,"publisher, developer, critic_score, total_sale..."
128,Murakumo: Renegade Mech Pursuit,XB,2,16,"genre, publisher, developer, total_sales, nort..."
86,Heroes of the Storm,PC,2,15,"genre, publisher, total_sales, north_america_s..."
183,Syndicate,PC,2,15,"genre, publisher, developer, total_sales, nort..."
189,The Train Giant,PC,2,15,"genre, publisher, developer, total_sales, euro..."
199,Transformers: The Game,PS2,2,15,"publisher, developer, total_sales, north_ameri..."
200,Transformers: The Game,PS3,2,15,"publisher, developer, total_sales, north_ameri..."
211,Warriors Orochi 2,X360,2,15,"publisher, critic_score, total_sales, north_am..."



Ejemplo detallado del par con mas diferencias: Zoo Tycoon DS | DS


,game_title,platform,genre,publisher,developer,critic_score,total_sales,north_america_sales,japan_sales,europe_sales,...,release_date,last_update,release_year,release_month,release_decade,regional_sum,sales_diff,sales_consistency_flag,total_sales_final,has_any_sales
1527,Zoo Tycoon DS,DS,Strategy,THQ,Blue Fang Games,4.6,0.98,0.86,0.01,0.03,...,2005-10-11,NaN,2005.0,10.0,2000.0,0.97,0.01,inconsistente,0.98,True
39270,Zoo Tycoon DS,DS,Misc,Unknown,Unknown,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,consistente_o_no_comparable,NaN,False


## Nota
Si quieres, puedo agregar en este mismo notebook visualizaciones (heatmap o barras) para estos tres puntos.